In [4]:
import sys

import os

#  Tell the notebook where to find your 'src' folder
sys.path.append(os.path.abspath('../'))
from src.database import get_db_connection, populate_database


In [5]:
get_db_connection()

<connection object at 0x000001AF340DB560; dsn: 'user=admin password=xxx dbname=bank_reviews host=localhost', closed: 0>

In [10]:
import os
import sys

# Ensure the notebook can see the src folder
if os.getcwd().endswith('notebooks') or os.getcwd().endswith('notebook'):
    os.chdir('..')
sys.path.append(os.path.abspath('.')) # Make sure this is '.' if you already moved up!

# Import your powerful, modular function
from src.database import populate_database

# Point this to your final CSV from Task 2
final_csv_path = 'data/processed/all_banks_themes.csv'

# Run the entire pipeline with one command
populate_database(final_csv_path)

Opening database transaction pipeline...
Syncing bank metadata...
Inserting 1664 records safely...
🚀 Data successfully inserted via modular pipeline!


In [11]:
import os
import sys
import pandas as pd
import warnings

# Suppress pandas SQL warnings for a cleaner notebook output
warnings.filterwarnings('ignore', category=UserWarning)

# Tell the notebook where to find your 'src' folder
if os.getcwd().endswith('notebooks') or os.getcwd().endswith('notebook'):
    os.chdir('..')
sys.path.append(os.path.abspath('.'))

from src.database import get_db_connection

# --- 1. Define your SQL Queries ---

query_volume = """
    SELECT b.bank_name, COUNT(r.review_id) AS total_reviews
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id
    GROUP BY b.bank_name
    ORDER BY total_reviews DESC;
"""

query_rating = """
    SELECT b.bank_name, ROUND(AVG(r.rating), 2) AS average_rating
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id
    GROUP BY b.bank_name
    ORDER BY average_rating DESC;
"""

query_nulls = """
    SELECT 
        COUNT(*) FILTER (WHERE review_text IS NULL) AS missing_text,
        COUNT(*) FILTER (WHERE rating IS NULL) AS missing_rating,
        COUNT(*) FILTER (WHERE sentiment_label IS NULL) AS missing_sentiment,
        COUNT(*) FILTER (WHERE bank_id IS NULL) AS missing_bank_link,
        COUNT(*) FILTER (WHERE identified_theme IS NULL) AS missing_theme
    FROM reviews;
"""

# --- 2. Execute Queries and Display Results ---

print("Opening secure database connection...\n")

with get_db_connection() as conn:
    # Read SQL directly into Pandas DataFrames
    df_volume = pd.read_sql(query_volume, conn)
    df_rating = pd.read_sql(query_rating, conn)
    df_nulls = pd.read_sql(query_nulls, conn)

print("✅ QUALITY CHECK 1: Total Reviews per Bank")
display(df_volume)

print("\n✅ QUALITY CHECK 2: Average Rating per Bank")
display(df_rating)

print("\n✅ QUALITY CHECK 3: Null Value Audit (All should be 0)")
display(df_nulls)

Opening secure database connection...

✅ QUALITY CHECK 1: Total Reviews per Bank


,bank_name,total_reviews
0,Bank of Abyssinia,559
1,Dashen Bank,558
2,Commercial Bank of Ethiopia,547



✅ QUALITY CHECK 2: Average Rating per Bank


,bank_name,average_rating
0,Commercial Bank of Ethiopia,4.07
1,Dashen Bank,3.92
2,Bank of Abyssinia,3.50



✅ QUALITY CHECK 3: Null Value Audit (All should be 0)


,missing_text,missing_rating,missing_sentiment,missing_bank_link,missing_theme
0,0,0,0,0,0


# 📈 Executive Summary: Competitive Intelligence & Product Recommendations

Based on the thematic clustering (TF-IDF) and transformer-based sentiment analysis of 1,666 user reviews across Ethiopia's top three digital banking platforms, we have identified distinct competitive advantages and critical friction points for each institution. 

Below is the synthesized breakdown of key drivers, pain points, and strategic recommendations.

---

## 1. Commercial Bank of Ethiopia (CBE)
**Overall Sentiment Profile:** [Insert your general observation, e.g., High volume, polarized sentiment]

* **🟢 Key Satisfaction Drivers:** * **Ubiquity & Utility:** Users frequently praise the app for basic, essential utilities like airtime top-ups and utility bill payments.
    * **Trust & Reliability:** The brand carries weight; when the app works, users appreciate the convenience of avoiding long physical branch queues.
* **🔴 Primary Pain Points:**
    * **System Stability:** High frequency of keywords like `"crash"`, `"update"`, and `"won't open"`. Users report severe instability following recent version updates.
    * **OTP Latency:** Significant negative sentiment is tied to `"SMS"` and `"code"` delivery failures, completely blocking user access.

**💡 Concrete Recommendations for CBE Product Team:**
1.  **Transition to In-App Push Authenticator:** To bypass the unreliable local SMS gateway infrastructure causing OTP delays, introduce secure Push Notification approvals or a time-based in-app authenticator token.
2.  **Staged Rollouts & Rollbacks:** The data shows sentiment tanks immediately after updates. Implement canary releases (releasing updates to 5% of users first) to catch crash loops before they hit the entire user base.

---

## 2. Bank of Abyssinia (BOA)
**Overall Sentiment Profile:** [Insert your general observation, e.g., Generally positive but dragged down by customer service complaints]

* **🟢 Key Satisfaction Drivers:**
    * **UI/UX Modernity:** Themes around `"design"`, `"easy"`, and `"smooth"` indicate that BOA currently holds the competitive edge in user interface and overall navigation.
    * **Feature Richness:** Positive mentions of digital wallet integrations and modern payment features.
* **🔴 Primary Pain Points:**
    * **Support Bottlenecks:** Keywords like `"customer service"`, `"call"`, and `"wait"` dominate the negative clusters. Users are frustrated by the inability to resolve digital issues without visiting a physical branch.
    * **Unclear Fee Structures:** Some negative sentiment clusters around `"charges"` and `"deducted"`, suggesting users feel blind-sided by transaction fees.

**💡 Concrete Recommendations for BOA Product Team:**
1.  **Deploy In-App Tier 1 Chat Support:** Integrate a conversational AI or live-chat ticketing system directly into the app to triage basic account issues, drastically reducing the load on phone support and branch staff.
2.  **Pre-Transaction Transparency:** Implement a clear "Fee Preview" screen that explicitly states the transaction cost *before* the user swipes to confirm the transfer.

---

## 3. Dashen Bank
**Overall Sentiment Profile:** [Insert your general observation, e.g., Moderate volume with a high concentration of login-related friction]

* **🟢 Key Satisfaction Drivers:**
    * **Ecosystem Integration:** High praise for Amole wallet integration and ease of merchant payments.
    * **International Utility:** Positive sentiment linked to diaspora features and remittance tracking.
* **🔴 Primary Pain Points:**
    * **Biometric Failures:** A massive spike in negative keywords like `"fingerprint"`, `"face"`, and `"password"`. Users are repeatedly locked out when biometrics fail to trigger.
    * **Account Recovery Loop:** Users complain that once locked out, the digital password reset flow is broken, forcing a physical branch visit.

**💡 Concrete Recommendations for Dashen Bank Product Team:**
1.  **Biometric API Overhaul & Fallback:** Audit the biometrics integration (specifically on older Android models) and ensure a seamless, immediate fallback to PIN/Password if the biometric sensor times out after 2 seconds.
2.  **Automated Digital Account Recovery:** Launch a fully digital, self-serve account recovery flow using security questions and automated ID verification to unblock users without requiring physical branch intervention.